DATASET: https://www.kaggle.com/datasets/andrewmvd/sp-500-stocks




# Below we will load in the datasets without downloading all the three csv files

In [391]:
import kagglehub
import os
# Download latest version
path = kagglehub.dataset_download("andrewmvd/sp-500-stocks")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'sp-500-stocks' dataset.
Path to dataset files: /kaggle/input/sp-500-stocks


# Below we will import all necessary libraries as well as machine learning models we will need.

In [392]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import shap
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import mean_squared_error as MSE
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.impute import KNNImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, f1_score, accuracy_score, recall_score, roc_curve, auc, precision_score, roc_auc_score
from sklearn.metrics import precision_recall_curve
import xgboost as xgb
from sklearn.metrics import ConfusionMatrixDisplay
from xgboost import XGBClassifier
import lightgbm as lgb
from lightgbm import LGBMClassifier
from sklearn.tree import DecisionTreeClassifier, plot_tree

In [393]:
# Loading the files
sp500_stocks = os.path.join(path, "sp500_stocks.csv")
sp_stocks = pd.read_csv(sp500_stocks)

sp500_index = os.path.join(path, "sp500_index.csv")
sp_index = pd.read_csv(sp500_index)

sp500_companies = os.path.join(path, "sp500_companies.csv")
sp_companies = pd.read_csv(sp500_companies)

# Identifying and cleaning data specifically for 'sp_index' below

In [394]:
# print all columns for index file
sp_index.columns

Index(['Date', 'S&P500'], dtype='object')

In [395]:
sp_index.rename(columns = {
    'Date': "entryDate",
    'S&P500': "S&P500_index_value"
}, inplace=True)

In [396]:
sp_index.dtypes

,0
entryDate,object
S&P500_index_value,float64


In [397]:
# Descriptions of bottom 5 and top 5 rows for index file
sp_index

,entryDate,S&P500_index_value
0,2014-12-22,2078.54
1,2014-12-23,2082.17
2,2014-12-24,2081.88
3,2014-12-26,2088.77
4,2014-12-29,2090.57
...,...,...
2512,2024-12-16,6074.08
2513,2024-12-17,6050.61
2514,2024-12-18,5872.16
2515,2024-12-19,5867.08


In [398]:
sp_index.duplicated().sum()

np.int64(0)

In [399]:
sp_index_missing = sp_index.isnull().sum()
print(sp_index_missing)

entryDate             0
S&P500_index_value    0
dtype: int64


Sp_index luckily has no duplicated or missing values, we can proceed further with it

# Identifying and cleaning data specifically for 'sp_stocks' below

In [400]:
# print all columns for stocks file
sp_stocks.columns

Index(['Date', 'Symbol', 'Adj Close', 'Close', 'High', 'Low', 'Open',
       'Volume'],
      dtype='object')

In [401]:
sp_stocks.rename(columns = {
    'Date': "entryDate",
    'Symbol': "Company_Symbol_Or_Ticker",
    'Adj Close': 'Adjusted_Closing_Price',
    'Close': 'Closing_Price',
    'High': 'Highest_Price',
    'Low': 'Lowest_Price',
    'Open': 'Opening_Price',
    'Volume': 'Total_Volume'
}, inplace=True)

In [402]:
sp_stocks.dtypes

,0
entryDate,object
Company_Symbol_Or_Ticker,object
Adjusted_Closing_Price,float64
Closing_Price,float64
Highest_Price,float64
Lowest_Price,float64
Opening_Price,float64
Total_Volume,float64


In [403]:
# Descriptions of bottom 5 and top 5 rows for index file
sp_stocks

,entryDate,Company_Symbol_Or_Ticker,Adjusted_Closing_Price,Closing_Price,Highest_Price,Lowest_Price,Opening_Price,Total_Volume
0,2010-01-04,MMM,NaN,NaN,NaN,NaN,NaN,NaN
1,2010-01-05,MMM,NaN,NaN,NaN,NaN,NaN,NaN
2,2010-01-06,MMM,NaN,NaN,NaN,NaN,NaN,NaN
3,2010-01-07,MMM,NaN,NaN,NaN,NaN,NaN,NaN
4,2010-01-08,MMM,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...
1891531,2024-12-16,ZTS,NaN,NaN,NaN,NaN,NaN,NaN
1891532,2024-12-17,ZTS,NaN,NaN,NaN,NaN,NaN,NaN
1891533,2024-12-18,ZTS,NaN,NaN,NaN,NaN,NaN,NaN
1891534,2024-12-19,ZTS,NaN,NaN,NaN,NaN,NaN,NaN


In [404]:
sp_stocks.duplicated().sum()

np.int64(0)

In [405]:
sp_stocks_missing = sp_stocks.isnull().sum()
print(sp_stocks_missing)

entryDate                         0
Company_Symbol_Or_Ticker          0
Adjusted_Closing_Price      1273705
Closing_Price               1273705
Highest_Price               1273705
Lowest_Price                1273705
Opening_Price               1273705
Total_Volume                1273705
dtype: int64


In [406]:
sp_stocks_missing_pct = (sp_stocks_missing / len(sp_stocks) * 100).round(2)
print(sp_stocks_missing_pct)

entryDate                    0.00
Company_Symbol_Or_Ticker     0.00
Adjusted_Closing_Price      67.34
Closing_Price               67.34
Highest_Price               67.34
Lowest_Price                67.34
Opening_Price               67.34
Total_Volume                67.34
dtype: float64


In [407]:
# Drop all rows in sp_stocks where any column contains missing values
sp_stocks.dropna(inplace=True)

# Verify the result
print("Shape of sp_stocks after dropping missing values:", sp_stocks.shape)
print("\nRemaining missing values in sp_stocks:")
print(sp_stocks.isnull().sum())

Shape of sp_stocks after dropping missing values: (617831, 8)

Remaining missing values in sp_stocks:
entryDate                   0
Company_Symbol_Or_Ticker    0
Adjusted_Closing_Price      0
Closing_Price               0
Highest_Price               0
Lowest_Price                0
Opening_Price               0
Total_Volume                0
dtype: int64


# Identifying and cleaning data specifically for 'sp_companies' below

In [408]:
# print all columns for companies file
sp_companies.columns

Index(['Exchange', 'Symbol', 'Shortname', 'Longname', 'Sector', 'Industry',
       'Currentprice', 'Marketcap', 'Ebitda', 'Revenuegrowth', 'City', 'State',
       'Country', 'Fulltimeemployees', 'Longbusinesssummary', 'Weight'],
      dtype='object')

In [409]:
sp_companies.rename(columns = {
    'Exchange': 'Exchange_Where_stocks_negotiated',
    'Symbol': 'Stock_Symbol',
    'Shortname': 'Company_Short_Name',
    'Longname': 'Company_Long_Name',
    'Sector': 'Company_Operates_Sector',
    'Industry': 'Industry_In_Sector_Where_Operates',
    'Currentprice': 'Current_Stock_price',
    'Marketcap': "current_Market_cap",
    'Ebitda': 'Earning_Before_Interest_Taxes_and_Depreciation',
    'Revenuegrowth': 'Revenue_Growth',
    'City': 'Matrix_City',
    'State': 'Matrix_State',
    'Country': 'Matrix_Country',
    'Fulltimeemployees': 'Number_of_Full_Time_Employees',
    'Longbusinesssummary': 'summary_Of_Company',
    'Weight': 'Percent_Of_Participation_on_Index'

})

,Exchange_Where_stocks_negotiated,Stock_Symbol,Company_Short_Name,Company_Long_Name,Company_Operates_Sector,Industry_In_Sector_Where_Operates,Current_Stock_price,current_Market_cap,Earning_Before_Interest_Taxes_and_Depreciation,Revenue_Growth,Matrix_City,Matrix_State,Matrix_Country,Number_of_Full_Time_Employees,summary_Of_Company,Percent_Of_Participation_on_Index
0,NMS,AAPL,Apple Inc.,Apple Inc.,Technology,Consumer Electronics,254.49,3846819807232,1.346610e+11,0.061,Cupertino,CA,United States,164000.0,"Apple Inc. designs, manufactures, and markets ...",0.069209
1,NMS,NVDA,NVIDIA Corporation,NVIDIA Corporation,Technology,Semiconductors,134.70,3298803056640,6.118400e+10,1.224,Santa Clara,CA,United States,29600.0,NVIDIA Corporation provides graphics and compu...,0.059350
2,NMS,MSFT,Microsoft Corporation,Microsoft Corporation,Technology,Software - Infrastructure,436.60,3246068596736,1.365520e+11,0.160,Redmond,WA,United States,228000.0,Microsoft Corporation develops and supports so...,0.058401
3,NMS,AMZN,"Amazon.com, Inc.","Amazon.com, Inc.",Consumer Cyclical,Internet Retail,224.92,2365033807872,1.115830e+11,0.110,Seattle,WA,United States,1551000.0,"Amazon.com, Inc. engages in the retail sale of...",0.042550
4,NMS,GOOGL,Alphabet Inc.,Alphabet Inc.,Communication Services,Internet Content & Information,191.41,2351625142272,1.234700e+11,0.151,Mountain View,CA,United States,181269.0,Alphabet Inc. offers various products and plat...,0.042309
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
497,NMS,CZR,"Caesars Entertainment, Inc.","Caesars Entertainment, Inc.",Consumer Cyclical,Resorts & Casinos,32.82,6973593600,3.668000e+09,-0.040,Reno,NV,United States,51000.0,"Caesars Entertainment, Inc. operates as a gami...",0.000125
498,NYQ,BWA,BorgWarner Inc.,BorgWarner Inc.,Consumer Cyclical,Auto Parts,31.88,6972155904,1.882000e+09,-0.048,Auburn Hills,MI,United States,39900.0,"BorgWarner Inc., together with its subsidiarie...",0.000125
499,NMS,QRVO,"Qorvo, Inc.","Qorvo, Inc.",Technology,Semiconductors,70.85,6697217024,6.731300e+08,-0.052,Greensboro,NC,United States,8700.0,"Qorvo, Inc. engages in development and commerc...",0.000120
500,NYQ,FMC,FMC Corporation,FMC Corporation,Basic Materials,Agricultural Inputs,50.15,6260525568,7.033000e+08,0.085,Philadelphia,PA,United States,5800.0,"FMC Corporation, an agricultural sciences comp...",0.000113


In [410]:
# Descriptions of bottom 5 and top 5 rows for companies file
sp_companies

,Exchange,Symbol,Shortname,Longname,Sector,Industry,Currentprice,Marketcap,Ebitda,Revenuegrowth,City,State,Country,Fulltimeemployees,Longbusinesssummary,Weight
0,NMS,AAPL,Apple Inc.,Apple Inc.,Technology,Consumer Electronics,254.49,3846819807232,1.346610e+11,0.061,Cupertino,CA,United States,164000.0,"Apple Inc. designs, manufactures, and markets ...",0.069209
1,NMS,NVDA,NVIDIA Corporation,NVIDIA Corporation,Technology,Semiconductors,134.70,3298803056640,6.118400e+10,1.224,Santa Clara,CA,United States,29600.0,NVIDIA Corporation provides graphics and compu...,0.059350
2,NMS,MSFT,Microsoft Corporation,Microsoft Corporation,Technology,Software - Infrastructure,436.60,3246068596736,1.365520e+11,0.160,Redmond,WA,United States,228000.0,Microsoft Corporation develops and supports so...,0.058401
3,NMS,AMZN,"Amazon.com, Inc.","Amazon.com, Inc.",Consumer Cyclical,Internet Retail,224.92,2365033807872,1.115830e+11,0.110,Seattle,WA,United States,1551000.0,"Amazon.com, Inc. engages in the retail sale of...",0.042550
4,NMS,GOOGL,Alphabet Inc.,Alphabet Inc.,Communication Services,Internet Content & Information,191.41,2351625142272,1.234700e+11,0.151,Mountain View,CA,United States,181269.0,Alphabet Inc. offers various products and plat...,0.042309
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
497,NMS,CZR,"Caesars Entertainment, Inc.","Caesars Entertainment, Inc.",Consumer Cyclical,Resorts & Casinos,32.82,6973593600,3.668000e+09,-0.040,Reno,NV,United States,51000.0,"Caesars Entertainment, Inc. operates as a gami...",0.000125
498,NYQ,BWA,BorgWarner Inc.,BorgWarner Inc.,Consumer Cyclical,Auto Parts,31.88,6972155904,1.882000e+09,-0.048,Auburn Hills,MI,United States,39900.0,"BorgWarner Inc., together with its subsidiarie...",0.000125
499,NMS,QRVO,"Qorvo, Inc.","Qorvo, Inc.",Technology,Semiconductors,70.85,6697217024,6.731300e+08,-0.052,Greensboro,NC,United States,8700.0,"Qorvo, Inc. engages in development and commerc...",0.000120
500,NYQ,FMC,FMC Corporation,FMC Corporation,Basic Materials,Agricultural Inputs,50.15,6260525568,7.033000e+08,0.085,Philadelphia,PA,United States,5800.0,"FMC Corporation, an agricultural sciences comp...",0.000113


In [411]:
sp_companies.dtypes

,0
Exchange,object
Symbol,object
Shortname,object
Longname,object
Sector,object
Industry,object
Currentprice,float64
Marketcap,int64
Ebitda,float64
Revenuegrowth,float64


In [412]:
sp_companies.duplicated().sum()

np.int64(0)

In [413]:
sp_companies_missing = sp_companies.isnull().sum()
print(sp_companies_missing)

Exchange                0
Symbol                  0
Shortname               0
Longname                0
Sector                  0
Industry                0
Currentprice            0
Marketcap               0
Ebitda                 29
Revenuegrowth           3
City                    0
State                  20
Country                 0
Fulltimeemployees       9
Longbusinesssummary     0
Weight                  0
dtype: int64


In [414]:
sp_companies_missing_pct = (sp_companies_missing / len(sp_companies) * 100).round(2)
print(sp_companies_missing_pct)

Exchange               0.00
Symbol                 0.00
Shortname              0.00
Longname               0.00
Sector                 0.00
Industry               0.00
Currentprice           0.00
Marketcap              0.00
Ebitda                 5.78
Revenuegrowth          0.60
City                   0.00
State                  3.98
Country                0.00
Fulltimeemployees      1.79
Longbusinesssummary    0.00
Weight                 0.00
dtype: float64


In [415]:
# Fill missing values in sp_companies
# Numerical columns: Ebitda, Revenuegrowth, Fulltimeemployees -> Fill with median
for col in ['Ebitda', 'Revenuegrowth', 'Fulltimeemployees']:
    if col in sp_companies.columns:
        median_val = sp_companies[col].median()
        sp_companies[col] = sp_companies[col].fillna(median_val)

# Categorical columns: State -> Fill with 'Unknown'
if 'State' in sp_companies.columns:
    sp_companies['State'] = sp_companies['State'].fillna('Unknown')

# Verify that there are no missing values left in sp_companies
print("Missing values remaining in sp_companies:")
print(sp_companies.isnull().sum())

Missing values remaining in sp_companies:
Exchange               0
Symbol                 0
Shortname              0
Longname               0
Sector                 0
Industry               0
Currentprice           0
Marketcap              0
Ebitda                 0
Revenuegrowth          0
City                   0
State                  0
Country                0
Fulltimeemployees      0
Longbusinesssummary    0
Weight                 0
dtype: int64
